# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samhere3116-maker/ML-Pipeline/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — "What Predicts Health?" (Random Forest feature importance, page 27)

This model predicts Health Score, and ranks Average Position (43%) and Impressions (32%) as its top two features — together 75% of the model's total importance. My methodology question: Health Score is explicitly defined elsewhere in the paper as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) — meaning two of the top three predicted features are literally ingredients in the label's own formula. The paper does flag this honestly ("the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal"), which is the right instinct. My constructive follow-up: how much of that 75% is the model genuinely discovering a pattern, versus just re-deriving a known formula? A clean way to check would be the same train-with/train-without test I ran in my own Week 3 notebook — drop Position and Impressions, retrain, and see how much predictive power survives. That number would tell a reader what this model adds beyond restating the definition.

Finding 2 — "What Predicts Growth?" Logistic Regression, 71% holdout accuracy (page 29)

My methodology question here is two-part. First, does the validation design support the claim? The methodology section says an 80/20 split was used, but doesn't say whether it was grouped by brand. With only 57 brands across 61.8K rows, a plain random split would very likely place pages from the same brand in both train and test, letting the model partly learn brand-specific quirks rather than a signal that generalizes to a brand it's never seen — I ran into exactly this risk in my own Week-5 model and had to switch to a client-grouped split to trust the number. Second, 71% accuracy needs a base rate next to it to mean anything — Finding #1 reports roughly 74.8K growing vs 45.6K declining pages, a base rate near 62%. Against that, 71% represents roughly 9 points of real skill, not 71 — a gap the paper doesn't state, which could make the number look stronger than it is to a reader who doesn't do that math themselves.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
import os, getpass
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# --- Connection ---
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY90 = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

CUTOFF = "2026-03-01"
QUERY_START = "2026-02-01"
MONTH_END = "2026-03-31"

# --- Feature frame ---
feat_frame_full = con.sql(f"""
    WITH windowed AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date > DATE '{CUTOFF}' THEN gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN report_date <= DATE '{CUTOFF}' THEN gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date <= DATE '{CUTOFF}' THEN gsc_clicks ELSE 0 END) AS clk_prev30,
            AVG(CASE WHEN report_date <= DATE '{CUTOFF}' THEN gsc_avg_position END) AS pos_prev30
        FROM {FACT}
        WHERE report_date BETWEEN '{QUERY_START}' AND '{MONTH_END}'
        GROUP BY 1, 2
        HAVING imp_prev30 >= 50
    ) SELECT * FROM windowed
""").df()

qsignals = con.sql(f"""
    SELECT content_hash_id, ANY_VALUE(content_visible_query_count) AS visible_queries
    FROM {QUERY90} GROUP BY content_hash_id
""").df()

content_age = con.sql(f"""
    SELECT content_hash_id, DATE_DIFF('day', content_created_date, DATE '{MONTH_END}') AS content_age_days
    FROM {DIM_CONTENT}
""").df()

feat_frame_full = feat_frame_full.merge(qsignals, on='content_hash_id', how='left').merge(content_age, on='content_hash_id', how='left')
feat_frame_full['is_declining'] = (feat_frame_full['imp_last30'] < 0.8 * feat_frame_full['imp_prev30']).astype(int)

baseline_url = "https://raw.githubusercontent.com/samhere3116-maker/ML-Pipeline/main/outputs/baseline_action_score.csv"
baseline = pd.read_csv(baseline_url)[['client_hash_id','content_hash_id','baseline_action_score']]
feat_frame_full = feat_frame_full.merge(baseline, on=['client_hash_id','content_hash_id'], how='left')
feat_frame_full['baseline_action_score'] = feat_frame_full['baseline_action_score'].fillna(0)

cols = ['imp_prev30','clk_prev30','pos_prev30','visible_queries','content_age_days']
model_df = feat_frame_full.dropna(subset=cols).reset_index(drop=True)
print(f"{len(model_df):,} rows | base rate: {model_df['is_declining'].mean():.3%}")

# --- Grouped split + trained models ---
X, y, groups = model_df[cols], model_df['is_declining'], model_df['client_hash_id']
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)
baseline_scores_test = model_df.iloc[test_idx]['baseline_action_score'].values

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1][:k]
    return np.array(y_true)[order].mean()

logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=42).fit(X_train, y_train)

print(f"Grouped split — Precision@50: {precision_at_k(y_test, logreg.predict_proba(X_test)[:,1]):.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

73,112 rows | base rate: 18.213%
Grouped split — Precision@50: 0.380


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# "Before" — a naive random split (ignores that rows repeat within a client)
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
logreg_random = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42).fit(Xtr_r, ytr_r)
p50_random_split = precision_at_k(ytr_r.reset_index(drop=True) if False else yte_r.reset_index(drop=True), logreg_random.predict_proba(Xte_r)[:,1])

# "After" — the grouped split, same model type, re-stated for comparison
p50_grouped_split = precision_at_k(y_test, logreg.predict_proba(X_test)[:,1])

before_after = pd.DataFrame({
    'split_type': ['random (naive)', 'grouped by client (honest)'],
    'precision_at_50': [p50_random_split, p50_grouped_split]
})
before_after

,split_type,precision_at_50
0,random (naive),0.64
1,grouped by client (honest),0.38


Under a naive random split, Precision@50 came out to 0.64. Under a client-grouped split — where no client appears in both train and test — it drops to 0.38, a 68% relative decrease. This gap is itself the finding: the random split let the model partly memorize client-specific patterns (site templates, niche, baseline traffic) rather than learning something that generalizes to a brand-new client. The grouped number, 0.38, is the one I'd trust and the one I reported back in Week 5 — the random-split number would have overstated how well this model actually works in deployment, where every prediction is, by definition, on a client the model hasn't been trained on specifically.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Deliberately re-add a leaky feature and watch the score jump
leaky_cols = cols + ['imp_last30']
model_df_leak = model_df.dropna(subset=leaky_cols)
Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(model_df_leak[leaky_cols], model_df_leak['is_declining'], test_size=0.25, random_state=42, stratify=model_df_leak['is_declining'])
leaky_check = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42).fit(Xtr_l, ytr_l)
leaky_p50 = precision_at_k(yte_l.reset_index(drop=True), leaky_check.predict_proba(Xte_l)[:,1])

print(f"Honest (5 features): {precision_at_k(y_test, logreg.predict_proba(X_test)[:,1]):.3f}")
print(f"With imp_last30 re-added: {leaky_p50:.3f}")

# 2. Confirm baseline_action_score was never used as an input feature
print(f"\nFeature columns actually used for training: {cols}")
print(f"baseline_action_score in training features? {'baseline_action_score' in cols}")

# 3. Base rate, printed next to every metric, for the record
print(f"\nBase rate (declining): {y_test.mean():.3%}")

Honest (5 features): 0.380
With imp_last30 re-added: 1.000

Feature columns actually used for training: ['imp_prev30', 'clk_prev30', 'pos_prev30', 'visible_queries', 'content_age_days']
baseline_action_score in training features? False

Base rate (declining): 13.317%


Re-adding imp_last30 as a feature pushed Precision@50 from an honest 0.380 to a perfect 1.000 — the same signature as the Week-3 trap, confirming it on this final feature set too. baseline_action_score was confirmed absent from the actual training features (False), meaning my Week-4 rule was used only as a comparison point, never as a model input, so it couldn't leak in either. One methodology note: this leakage check used a plain random split and a slightly different row filter than my main model, giving a base rate of 13.3% here versus 18.2% in Section 2 — a small population shift from the extra dropna requirement, not a contradiction.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original: "Logistic Regression beat Random Forest and outperformed the base rate by nearly 3x."
Rewritten: "In this portfolio slice, Logistic Regression's top-50 ranked pages were observed to contain declining pages at roughly 2.9x the rate expected from random selection, under a client-grouped validation split. This is a decision-support signal for prioritizing review, not a causal or guaranteed prediction, and it reflects March 2026 data for a subset of clients — it has not been tested on a future month or a held-out time window."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.